# RRN — CPU verification, then GPU training from a local full LMDB
Run the CPU verification first without allocating a GPU. Only after it passes, switch to T4, copy the compact LMDB once to `/content`, and run the smoke test. Checkpoints and metrics are written directly to Drive.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!pip -q install lmdb pyspng

In [ ]:
from pathlib import Path
import subprocess, torch
print('Runtime:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')
PROJECT = Path('/content/drive/MyDrive/train_ai/projects/RNN')
DRIVE_DATA = Path('/content/drive/MyDrive/train_ai/datasets/vimeo90k/vimeo90k_septuplet_compact.lmdb')
LOCAL_DATA = Path('/content/vimeo90k_septuplet_compact.lmdb')
ADAPTER = Path('/content/rrn-colab-adapter')
UPSTREAM = Path('/content/RRN-upstream')
UPSTREAM_COMMIT = 'db440c5d363dfe6c11bdd6c82e63781b295e6211'
if not ADAPTER.exists():
    subprocess.run(['git', 'clone', '--depth', '1', 'https://github.com/koorock92/rrn-colab-adapter.git', str(ADAPTER)], check=True)
else:
    subprocess.run(['git', '-C', str(ADAPTER), 'fetch', 'origin', 'main'], check=True)
    subprocess.run(['git', '-C', str(ADAPTER), 'reset', '--hard', 'origin/main'], check=True)
if not UPSTREAM.exists():
    subprocess.run(['git', 'clone', 'https://github.com/junpan19/RRN.git', str(UPSTREAM)], check=True)
subprocess.run(['git', '-C', str(UPSTREAM), 'checkout', UPSTREAM_COMMIT], check=True)
WORK = ADAPTER / 'src'
SMOKE_OUT = PROJECT / 'logs/full_lmdb_gpu_smoke'
FULL_OUT = PROJECT / 'logs/full_lmdb_gpu_train'
!df -h / /content/drive

## Phase 1 — CPU verification (no GPU required)
Checks the Drive LMDB size and entries, decodes one real Vimeo clip, runs one full RRN forward pass, and verifies checkpoint serialization. It does not copy the 83 GiB database.

In [ ]:
import sys, lmdb, torch, tempfile
EXPECTED_DATA_BYTES = 89_132_445_696
EXPECTED_ENTRIES = 641_909
source_file = DRIVE_DATA / 'data.mdb'
assert source_file.is_file(), source_file
assert source_file.stat().st_size == EXPECTED_DATA_BYTES, source_file.stat().st_size
env = lmdb.open(str(DRIVE_DATA), readonly=True, lock=False, readahead=False, subdir=True)
try:
    entries = env.stat()['entries']
finally:
    env.close()
assert entries == EXPECTED_ENTRIES, entries
sys.path.insert(0, str(WORK))
from vimeo_sharded_dataset import VimeoSeptupletDataset
from upstream_rrn_adapter import build_rrn
dataset = VimeoSeptupletDataset(DRIVE_DATA, split='train', scale=4, crop_size=64, max_samples=1, augment=False, decoder='auto')
if dataset.env is not None:
    dataset.env.close(); dataset.env = None
low_resolution, target, name = dataset[0]
if dataset.env is not None:
    dataset.env.close(); dataset.env = None
model = build_rrn(UPSTREAM, 4, 128, 10).cpu().eval()
with torch.inference_mode():
    prediction = model(low_resolution.unsqueeze(0))
assert prediction.shape == target.unsqueeze(0).shape, (prediction.shape, target.shape)
assert torch.isfinite(prediction).all()
checkpoint = Path(tempfile.gettempdir()) / 'rrn_cpu_verify.pt'
torch.save({'model': model.state_dict(), 'sample': name, 'entries': entries}, checkpoint)
loaded = torch.load(checkpoint, map_location='cpu', weights_only=False)
assert loaded['sample'] == name and loaded['entries'] == EXPECTED_ENTRIES
print(f'CPU_VERIFY_PASSED sample={name} decoder={dataset.decoder} entries={entries} output={tuple(prediction.shape)}')

## Phase 2 — switch to T4, then copy and verify the complete LMDB
Do not run this cell on CPU. Change the runtime to T4 only after `CPU_VERIFY_PASSED`. The copy is resumable within the GPU runtime.

In [ ]:
import lmdb, shutil, subprocess, torch
assert torch.cuda.is_available(), 'CPU 驗證完成後，請先切換至 T4 GPU runtime'
EXPECTED_DATA_BYTES = 89_132_445_696
EXPECTED_ENTRIES = 641_909
source_file = DRIVE_DATA / 'data.mdb'
assert source_file.is_file(), source_file
assert source_file.stat().st_size == EXPECTED_DATA_BYTES, source_file.stat().st_size
free = shutil.disk_usage('/content').free
assert free > EXPECTED_DATA_BYTES + 20 * 2**30, f'Insufficient VM disk: {free / 2**30:.1f} GiB free'
local_file = LOCAL_DATA / 'data.mdb'
if not local_file.is_file() or local_file.stat().st_size != EXPECTED_DATA_BYTES:
    LOCAL_DATA.mkdir(parents=True, exist_ok=True)
    cmd = ['rsync', '-ah', '--partial', '--info=progress2', f'{DRIVE_DATA}/', f'{LOCAL_DATA}/']
    print('Copying full LMDB to VM local disk...', flush=True)
    subprocess.run(cmd, check=True)
else:
    print('Local LMDB already complete; copy skipped.')
assert local_file.stat().st_size == EXPECTED_DATA_BYTES
env = lmdb.open(str(LOCAL_DATA), readonly=True, lock=False, readahead=False, subdir=True)
try:
    entries = env.stat()['entries']
finally:
    env.close()
assert entries == EXPECTED_ENTRIES, entries
print(f'LOCAL_LMDB_READY entries={entries} size={local_file.stat().st_size / 2**30:.2f} GiB')
!df -h /

## 20-step smoke test
Runs the real RRN model, local full LMDB, fast decoding, two workers, GPU downsampling, AMP, validation, and Drive checkpoints.

In [ ]:
import subprocess, sys
last = SMOKE_OUT / 'last.pt'
cmd = [sys.executable, '-u', str(WORK / 'train_rrn_full_lmdb.py'),
       '--upstream-dir', str(UPSTREAM), '--data', str(LOCAL_DATA), '--output', str(SMOKE_OUT),
       '--scale', '4', '--crop-size', '64', '--batch-size', '16', '--workers', '2',
       '--channels', '128', '--blocks', '10', '--epochs', '70', '--save-every', '10',
       '--metrics-flush-every', '10', '--log-every', '1', '--max-steps', '20',
       '--val-samples', '100', '--decoder', 'auto', '--prefetch-factor', '8',
       '--gpu-downsample', '--amp']
if last.exists():
    cmd += ['--resume', str(last)]
print(' '.join(cmd), flush=True)
process = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
for line in process.stdout:
    print(line, end='', flush=True)
if process.wait() != 0:
    raise subprocess.CalledProcessError(process.returncode, cmd)

In [ ]:
import json
lines = (SMOKE_OUT / 'metrics.jsonl').read_text().splitlines()
recent = [json.loads(line) for line in lines[-20:]]
print('steps:', len(recent))
print('mean wall seconds/step:', sum(x['wall_seconds'] for x in recent) / len(recent))
print('mean data seconds/step:', sum(x['data_seconds'] for x in recent) / len(recent))
print('last:', (SMOKE_OUT / 'last.pt').exists())
print('best:', (SMOKE_OUT / 'best.pt').exists())
!nvidia-smi --query-gpu=name,memory.used,utilization.gpu --format=csv

## Full training with a live dashboard
Leave disabled until the smoke-test speed and checkpoints have been verified. `last.pt` resumes all training state; `best.pt` is selected by validation PSNR.

In [ ]:
CONFIRM_FULL_TRAINING = False
if not CONFIRM_FULL_TRAINING:
    print('尚未啟動正式訓練。先確認上方 smoke test 的速度與 checkpoint。')
else:
    import subprocess, sys, math
    from collections import deque
    from html import escape
    from IPython.display import HTML, display
    last = FULL_OUT / 'last.pt'
    cmd = [sys.executable, '-u', str(WORK / 'train_rrn_full_lmdb.py'),
           '--upstream-dir', str(UPSTREAM), '--data', str(LOCAL_DATA), '--output', str(FULL_OUT),
           '--scale', '4', '--crop-size', '64', '--batch-size', '16', '--workers', '2',
           '--channels', '128', '--blocks', '10', '--epochs', '70', '--save-every', '1000',
           '--metrics-flush-every', '100', '--log-every', '25', '--val-samples', '100',
           '--decoder', 'auto', '--prefetch-factor', '8', '--gpu-downsample', '--amp']
    if last.exists():
        cmd += ['--resume', str(last)]
    print(' '.join(cmd), flush=True)
    ui = {'epoch':'-', 'step':0, 'batch':0, 'batches':1, 'loss':'--', 'mae':'--', 'lr':'--',
          'psnr':'--', 'best':'--', 'data_s':'--', 'compute_s':'--', 'wall_s':'--',
          'samples_s':'--', 'eta':'--', 'checkpoint':0, 'event':'Starting trainer...'}
    recent_wall = deque(maxlen=20)
    status = display(HTML('Starting...'), display_id=True)
    def fields(line): return dict(token.split('=', 1) for token in line.split()[1:] if '=' in token)
    def metric(value, digits=4):
        try:
            number = float(value); return f'{number:.{digits}f}' if math.isfinite(number) else '--'
        except (TypeError, ValueError): return '--'
    def duration(seconds):
        seconds = max(0, int(seconds)); hours, seconds = divmod(seconds, 3600); minutes, seconds = divmod(seconds, 60)
        return f'{hours:02d}:{minutes:02d}:{seconds:02d}'
    def render(color='#22c55e'):
        percent = 100.0 * ui['batch'] / max(ui['batches'], 1)
        status.update(HTML(f"""<div style='font-family:ui-monospace,SFMono-Regular,Consolas,monospace;padding:16px;background:#111827;color:#e5e7eb;border-radius:10px;line-height:1.55'>
        <div style='font-size:15px;font-weight:700'>RRN ×4 · Epoch {escape(str(ui['epoch']))} · Step {ui['step']:,}</div>
        <div>PSNR: <b style='color:#60a5fa'>{ui['psnr']}</b> · Best: <b style='color:#a78bfa'>{ui['best']}</b> · MAE: {ui['mae']} · Loss: {ui['loss']} · LR: {ui['lr']}</div>
        <div style='height:14px;background:#374151;border-radius:7px;margin:9px 0 5px'><div style='height:14px;width:{percent:.2f}%;background:{color};border-radius:7px'></div></div>
        <div>{ui['batch']:,}/{ui['batches']:,} ({percent:.1f}%) · {ui['wall_s']} s/step · {ui['samples_s']} clips/s · ETA {ui['eta']}</div>
        <div style='color:#9ca3af'>Data {ui['data_s']}s · GPU compute {ui['compute_s']}s · checkpoint step {ui['checkpoint']:,}</div>
        <div style='margin-top:8px;color:#9ca3af;font-size:12px'>{escape(ui['event'])}</div></div>"""))
    process = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for raw in process.stdout:
        line = raw.strip()
        if not line: continue
        data = fields(line)
        if line.startswith('TRAIN '):
            ui.update(epoch=data['epoch'], step=int(data['step']), loss=data['loss'], mae=data['mae'], lr=data['lr'],
                      psnr=metric(data.get('val_psnr')), best=metric(data.get('best_psnr')), data_s=data['data_s'],
                      compute_s=data['compute_s'], wall_s=data['wall_s'], samples_s=data['samples_s'],
                      checkpoint=int(data.get('checkpoint_step', ui['checkpoint'])))
            ui['batch'], ui['batches'] = map(int, data['batch'].split('/'))
            recent_wall.append(float(data['wall_s']))
            ui['eta'] = duration((ui['batches'] - ui['batch']) * sum(recent_wall) / len(recent_wall))
        elif line.startswith('CHECKPOINT '):
            ui['checkpoint'] = int(data['step']); ui['event'] = f"Checkpoint saved to Drive at step {ui['checkpoint']:,}"
        elif line.startswith(('VALIDATION', 'EPOCH_COMPLETE', 'INPUT_PIPELINE', 'RESUMED')):
            ui['event'] = line
            if 'psnr' in data: ui['psnr'] = metric(data['psnr'])
            if 'best_psnr' in data: ui['best'] = metric(data['best_psnr'])
        else: print(line, flush=True)
        render()
    returncode = process.wait()
    if returncode:
        render('#dc2626'); raise subprocess.CalledProcessError(returncode, cmd)
    ui['event'] = 'Training finished successfully.'; render('#16a34a')